In [1]:
import os
import json
from pathlib import Path

import cv2
import numpy as np
from tqdm import tqdm

# ==========================
# CONFIGURACIÓN GENERAL
# ==========================

# Rutas de entrada (big images + JSON COCO)
TRAIN_IMG_DIR = "general_dataset/general_dataset/train"
TRAIN_JSON = "general_dataset/general_dataset/groundtruth/json/big_size/train_big_size_A_B_E_K_WH_WB.json"

VAL_IMG_DIR = "general_dataset/general_dataset/val"
VAL_JSON = "general_dataset/general_dataset/groundtruth/json/big_size/val_big_size_A_B_E_K_WH_WB.json"

# Rutas de salida (tiles + labels YOLO)
OUT_BASE_DIR = "general_dataset/general_dataset/yolo_tiles"
TRAIN_OUT_IMG = os.path.join(OUT_BASE_DIR, "images/train")
TRAIN_OUT_LBL = os.path.join(OUT_BASE_DIR, "labels/train")
VAL_OUT_IMG = os.path.join(OUT_BASE_DIR, "images/val")
VAL_OUT_LBL = os.path.join(OUT_BASE_DIR, "labels/val")

os.makedirs(TRAIN_OUT_IMG, exist_ok=True)
os.makedirs(TRAIN_OUT_LBL, exist_ok=True)
os.makedirs(VAL_OUT_IMG, exist_ok=True)
os.makedirs(VAL_OUT_LBL, exist_ok=True)

# Parámetros de tiles
TILE_SIZE = 1024          # tamaño del recorte
OVERLAP = 0.25            # 25% de solape
MIN_COVERAGE = 0.3        # mínimo % del área original de la caja que debe quedar dentro del tile

# ==========================
# FUNCIONES AUXILIARES
# ==========================

def load_coco_json(json_path):
    """Carga anotaciones COCO y devuelve:
    - images: dict image_id -> {file_name, width, height}
    - annotations: dict image_id -> [ann, ...]
    - categories: dict cat_id -> cat_name
    """
    with open(json_path, "r") as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco["images"]}
    annotations = {}
    for ann in coco["annotations"]:
        img_id = ann["image_id"]
        annotations.setdefault(img_id, []).append(ann)

    categories = {cat["id"]: cat["name"] for cat in coco["categories"]}

    return images, annotations, categories


def get_class_mapping(categories):
    """Devuelve:
    - cat_id_to_cls: dict cat_id -> clase_idx (0..N-1)
    - cls_to_name: dict clase_idx -> nombre"""
    sorted_items = sorted(categories.items(), key=lambda x: x[0])  # ordenar por id de categoría
    cat_ids = [cid for cid, _ in sorted_items]
    names = [name for _, name in sorted_items]

    cat_id_to_cls = {cid: i for i, cid in enumerate(cat_ids)}
    cls_to_name = {i: n for i, n in enumerate(names)}

    return cat_id_to_cls, cls_to_name


def rotate_image_and_bboxes_90_clockwise(img, bboxes):
    """
    Rota la imagen 90° clockwise y actualiza las bboxes COCO [x, y, w, h].
    img: np.array (H, W, C)
    bboxes: lista de [x, y, w, h]
    Retorna: img_rot, bboxes_rot
    """
    h, w = img.shape[:2]
    img_rot = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)  # nuevo tamaño: (w, h)

    bboxes_rot = []
    for (x, y, bw, bh) in bboxes:
        # Coordenadas originales
        x1, y1 = x, y
        x2, y2 = x + bw, y + bh

        # Fórmulas para 90° clockwise (usando sistema continuo)
        x1p = h - (y2)
        y1p = x1
        x2p = h - (y1)
        y2p = x2

        # Normalizar para asegurar x_min < x_max, etc.
        x_min = min(x1p, x2p)
        y_min = min(y1p, y2p)
        x_max = max(x1p, x2p)
        y_max = max(y1p, y2p)

        bw_new = x_max - x_min
        bh_new = y_max - y_min

        bboxes_rot.append([x_min, y_min, bw_new, bh_new])

    return img_rot, bboxes_rot


def generate_tile_coords(w, h, tile_size, overlap):
    """Genera coordenadas superiores-izquierdas (x, y) de tiles con solape."""
    stride = int(tile_size * (1 - overlap))
    if stride <= 0:
        raise ValueError("OVERLAP demasiado grande. Debe ser <1.")

    def compute_starts(dim):
        if dim <= tile_size:
            return [0]
        starts = list(range(0, dim - tile_size + 1, stride))
        if starts[-1] != dim - tile_size:
            starts.append(dim - tile_size)
        return starts

    xs = compute_starts(w)
    ys = compute_starts(h)
    return xs, ys


def clip_bbox_to_tile(bbox, tile_x, tile_y, tile_size):
    """Recorta una bbox global [x, y, w, h] al rectángulo del tile.
    Retorna (new_bbox, coverage_ratio) o (None, 0) si no hay intersección suficiente.
    coverage_ratio = área_intersección / área_original
    """
    x, y, bw, bh = bbox
    x1, y1 = x, y
    x2, y2 = x + bw, y + bh

    tx1, ty1 = tile_x, tile_y
    tx2, ty2 = tile_x + tile_size, tile_y + tile_size

    ix1 = max(x1, tx1)
    iy1 = max(y1, ty1)
    ix2 = min(x2, tx2)
    iy2 = min(y2, ty2)

    iw = ix2 - ix1
    ih = iy2 - iy1

    if iw <= 1 or ih <= 1:
        return None, 0.0

    inter_area = iw * ih
    orig_area = bw * bh
    coverage = inter_area / (orig_area + 1e-6)

    if coverage < MIN_COVERAGE:
        return None, coverage

    # Coordenadas relativas al tile
    new_x = ix1 - tx1
    new_y = iy1 - ty1
    new_w = iw
    new_h = ih

    return [new_x, new_y, new_w, new_h], coverage


def coco_bbox_to_yolo(bbox, img_w, img_h):
    """Convierte bbox [x, y, w, h] en formato COCO a YOLO [cx, cy, w, h] normalizado."""
    x, y, w, h = bbox
    cx = x + w / 2.0
    cy = y + h / 2.0

    return [
        cx / img_w,
        cy / img_h,
        w / img_w,
        h / img_h,
    ]


def process_split(
    split_name,
    img_dir,
    json_path,
    out_img_dir,
    out_lbl_dir,
    cat_id_to_cls,
    rotate_vertical=True,
):
    """
    Procesa un split (train/val):
    - Carga imágenes grandes + anotaciones.
    - Rota a horizontal si h > w.
    - Genera tiles 1024x1024 con solape.
    - Guarda imágenes y labels YOLO.
    """
    images, annotations, categories = load_coco_json(json_path)

    all_image_ids = list(images.keys())
    print(f"[{split_name}] Imágenes en JSON: {len(all_image_ids)}")

    num_tiles_total = 0
    num_tiles_with_boxes = 0

    for img_id in tqdm(all_image_ids, desc=f"Procesando {split_name}"):
        img_info = images[img_id]
        file_name = img_info["file_name"]
        img_path = os.path.join(img_dir, file_name)

        if not os.path.exists(img_path):
            print(f"  [WARN] No se encontró la imagen: {img_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"  [WARN] Error leyendo imagen: {img_path}")
            continue

        h, w = img.shape[:2]

        # Bboxes originales de esta imagen
        anns = annotations.get(img_id, [])
        bboxes = [ann["bbox"] for ann in anns]
        cat_ids = [ann["category_id"] for ann in anns]

        # Rotar a horizontal si la imagen es vertical y se desea
        if rotate_vertical and h > w:
            img, bboxes = rotate_image_and_bboxes_90_clockwise(img, bboxes)
            h, w = img.shape[:2]

        # Tileo
        xs, ys = generate_tile_coords(w, h, TILE_SIZE, OVERLAP)

        # Construir stem base para nombres de archivo
        stem = Path(file_name).stem

        for yi, ty in enumerate(ys):
            for xi, tx in enumerate(xs):
                num_tiles_total += 1

                tile = img[ty:ty + TILE_SIZE, tx:tx + TILE_SIZE]
                tile_h, tile_w = tile.shape[:2]
                if tile_h != TILE_SIZE or tile_w != TILE_SIZE:
                    # Por seguridad, saltar tiles incompletos
                    continue

                # Ajustar bboxes al tile
                tile_yolo_boxes = []
                tile_classes = []

                for bbox, cat_id in zip(bboxes, cat_ids):
                    clipped, coverage = clip_bbox_to_tile(
                        bbox, tx, ty, TILE_SIZE
                    )
                    if clipped is None:
                        continue

                    yolo_bbox = coco_bbox_to_yolo(clipped, TILE_SIZE, TILE_SIZE)
                    cls_idx = cat_id_to_cls[cat_id]

                    # Evitar cajas degeneradas (w/h muy pequeñas o fuera de [0,1])
                    _, _, bw_n, bh_n = yolo_bbox
                    if bw_n <= 0 or bh_n <= 0:
                        continue

                    tile_yolo_boxes.append(yolo_bbox)
                    tile_classes.append(cls_idx)
                
                save_negative_tiles = False

                if len(tile_yolo_boxes) == 0 and not save_negative_tiles:
                    continue

                num_tiles_with_boxes += int(len(tile_yolo_boxes) > 0)

                # Nombre del tile
                tile_name = f"{stem}_x{tx}_y{ty}.jpg"
                tile_path = os.path.join(out_img_dir, tile_name)
                label_path = os.path.join(out_lbl_dir, tile_name.replace(".jpg", ".txt"))

                # Guardar imagen
                cv2.imwrite(tile_path, tile)

                # Guardar labels YOLO
                with open(label_path, "w") as lf:
                    for cls_idx, yb in zip(tile_classes, tile_yolo_boxes):
                        cx, cy, bw_n, bh_n = yb
                        lf.write(f"{cls_idx} {cx:.6f} {cy:.6f} {bw_n:.6f} {bh_n:.6f}\n")

    print(f"[{split_name}] Tiles totales generados (antes de filtro negativos): {num_tiles_total}")
    print(f"[{split_name}] Tiles guardados con al menos 1 bbox: {num_tiles_with_boxes}")


# ==========================
# EJECUCIÓN
# ==========================

# Primero cargamos train.json solo para obtener categorías
train_images, train_annotations, train_categories = load_coco_json(TRAIN_JSON)
cat_id_to_cls, cls_to_name = get_class_mapping(train_categories)

print("Mapeo de clases (cat_id -> idx -> nombre):")
for cid, cname in train_categories.items():
    print(f"  cat_id={cid:2d} -> cls={cat_id_to_cls[cid]} -> {cname}")

# Procesar train
process_split(
    split_name="train",
    img_dir=TRAIN_IMG_DIR,
    json_path=TRAIN_JSON,
    out_img_dir=TRAIN_OUT_IMG,
    out_lbl_dir=TRAIN_OUT_LBL,
    cat_id_to_cls=cat_id_to_cls,
    rotate_vertical=True,
)

# Procesar val
process_split(
    split_name="val",
    img_dir=VAL_IMG_DIR,
    json_path=VAL_JSON,
    out_img_dir=VAL_OUT_IMG,
    out_lbl_dir=VAL_OUT_LBL,
    cat_id_to_cls=cat_id_to_cls,
    rotate_vertical=True,
)

# ==========================
# CREAR data.yaml PARA YOLO
# ==========================

data_yaml_path = os.path.join(OUT_BASE_DIR, "data.yaml")

names_list = [cls_to_name[i] for i in range(len(cls_to_name))]

with open(data_yaml_path, "w") as f:
    f.write(f"path: {OUT_BASE_DIR}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("names:\n")
    for i, name in enumerate(names_list):
        f.write(f"  {i}: {name}\n")

print(f"\nArchivo data.yaml generado en: {data_yaml_path}")
print("Contenido de names:")
for i, name in enumerate(names_list):
    print(f"  {i}: {name}")


Mapeo de clases (cat_id -> idx -> nombre):
  cat_id= 1 -> cls=0 -> Alcelaphinae
  cat_id= 2 -> cls=1 -> Buffalo
  cat_id= 3 -> cls=2 -> Kob
  cat_id= 4 -> cls=3 -> Warthog
  cat_id= 5 -> cls=4 -> Waterbuck
  cat_id= 6 -> cls=5 -> Elephant
[train] Imágenes en JSON: 928


Procesando train: 100%|██████████| 928/928 [04:14<00:00,  3.65it/s]


[train] Tiles totales generados (antes de filtro negativos): 35795
[train] Tiles guardados con al menos 1 bbox: 4601
[val] Imágenes en JSON: 111


Procesando val: 100%|██████████| 111/111 [00:35<00:00,  3.15it/s]

[val] Tiles totales generados (antes de filtro negativos): 4265
[val] Tiles guardados con al menos 1 bbox: 670

Archivo data.yaml generado en: general_dataset/general_dataset/yolo_tiles\data.yaml
Contenido de names:
  0: Alcelaphinae
  1: Buffalo
  2: Kob
  3: Warthog
  4: Waterbuck
  5: Elephant
